# Explainability and Fraud Analyst Reason Codes

## Objective
This notebook explains why a transaction is predicted as risky. The goal is to translate model output into fraud analyst language: primary driver, probability impact, severity, reason code, and recommended action.


In [ ]:
# Import project utilities and analysis libraries.
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(REPO_ROOT / 'src'))

from fraud_pipeline import BASE_FEATURES, load_transactions, train_top_feature_model, predict_transaction


## Load Model And Data
The explainability workflow uses the same training pipeline as the dashboard. This keeps notebook evidence aligned with the deployed application.


In [ ]:
# Load data and train the same top-feature model used by the Streamlit app.
transactions = load_transactions(REPO_ROOT)
bundle = train_top_feature_model(transactions, top_n=10)

print('Selected model:', bundle.model_name)
print('Selected review threshold:', round(bundle.review_threshold, 2))
print('Top features:', bundle.top_features)


## Select A Fraud Example
To keep the explanation realistic, this notebook selects a real fraudulent transaction from the dataset and explains the model decision.


In [ ]:
# Select a real fraudulent transaction with high device risk.
fraud_examples = transactions[transactions['fraud_label'] == 1]
example_transaction = (
    fraud_examples.sort_values('device_risk_score', ascending=False)
    .iloc[0][BASE_FEATURES]
    .to_dict()
)

print(example_transaction)


## Generate Local Explanation
Each explanation compares the transaction value with a normal reference value. The model then estimates how much each feature changes the fraud probability.


In [ ]:
# Predict fraud risk and generate analyst-readable reason codes.
probability, label, drivers = predict_transaction(
    bundle,
    example_transaction,
    transactions,
)

print('Prediction:', label)
print('Fraud probability:', f'{probability:.2%}')

display(
    drivers[
        [
            'feature',
            'input_value',
            'reference_value',
            'probability_impact',
            'severity',
            'risk_direction',
            'analyst_reason',
            'recommended_action',
        ]
    ]
)


In [ ]:
# Visualise the local probability impact of each feature.
plot_df = drivers.sort_values('probability_impact', ascending=True)

plt.figure(figsize=(10, 6))
colors = ['#b91c1c' if value > 0 else '#2f6f73' for value in plot_df['probability_impact']]
plt.barh(plot_df['feature'], plot_df['probability_impact'], color=colors)
plt.axvline(0, color='black', linewidth=1)
plt.title('Local Feature Impact on Fraud Probability')
plt.xlabel('Change in fraud probability vs reference value')
plt.ylabel('Feature')
plt.show()


## Analyst Interpretation
The explanation gives both technical and operational value:

- `probability_impact` shows how much the feature pushed the prediction up or down.
- `severity` helps triage the reason code.
- `analyst_reason` explains the evidence in plain language.
- `recommended_action` turns the model output into an investigation step.

This is stronger than only showing a fraud label because it supports transparent and reviewable decision-making.
